# General Functions
> These are functions will apply to all the tables

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

In [0]:
def drop_url(df):
    if "url" in df.columns:
        df = df.drop("url")
    return df

In [0]:
def trim_strings(df):
    return df.select([
        F.trim(F.col(c)).alias(c) if df.schema[c].dataType == StringType()
        else F.col(c)
        for c in df.columns
    ])

In [0]:
def mapping(df, table_name):
    if table_name.startswith("race_events_"):
        status_map = {
            "R": "Resumed",
            "S": "Standing Start",
            "Y": "Rolling Start",
            "N": "Not Resumed"
        }
    else:
        status_map = {
            "R": "Retired",
            "D": "Disqualified",
            "E": "Excluded",
            "F": "Failed",
            "N": "Not Classified",
            "W": "Withdrawn",
            r"\N": None,
            "-": None,
            "USA": "United States",
            "UK": "United Kingdom",
            "UAE": "United Arab Emirates"
        }

    for col in df.columns:
        if df.schema[col].dataType == StringType():
            rename = F.col(col)
            for key, value in status_map.items():
                rename = F.when(F.col(col) == key, value).otherwise(rename)
            df = df.withColumn(col, rename)
    return df

In [0]:
def cast_date_columns(df):
    for col in df.columns:
        if ("date" in col or "dob" in col):
            df = df.withColumn(
                col,
                F.coalesce(
                    F.try_to_date(F.col(col), "yyyy-MM-dd"),
                    F.try_to_date(F.col(col), "MM/dd/yy"),
                    F.try_to_date(F.col(col), "M/d/yy"),
                    F.try_to_date(F.col(col), "M/dd/yy"),
                    F.try_to_date(F.col(col), "MM/d/yy"),
                )
            )
    return df

In [0]:
def cast_to_integer(df):
    target_columns = {"milliseconds", "position", "number", "grid", "rank"}  # year removed
    cols_in_df = target_columns.intersection(set(df.columns))
    for col_name in cols_in_df:
        df = df.withColumn(col_name, F.col(col_name).try_cast(IntegerType()))
    return df

In [0]:
#Master to apply functions
def apply_general_silver_transforms(df, table_name):
    df = drop_url(df)
    df = trim_strings(df)
    df = mapping(df, table_name)
    df = cast_date_columns(df)
    df = cast_to_integer(df)
    return df